In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import powerlaw as pwl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np


In [2]:
con = sqlite3.connect("vp_data2_isikud.db")
cur = con.cursor()
cur.execute('ATTACH DATABASE "v33.db" AS v33')

## 1. tabel 
## verb -> palju esineb obl+kääne (6 kohakäänet) : mitu matchi ja mitu distinct root 

Võtta välja kõik mis on kohakäändes ja sõna deprel on obl

In [187]:
%%time

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_wcomps
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes_wcomps as
SELECT distinct
    tr.head_id as head_id,
    tbl1.verb as verb,
    tbl1.verb_compound as verb_compound,
    tr.id as transaction_id,
    tr.lemma as root_word,
    tr.deprel as word_deprel,
    tr.pos as pos,
    tr.feats as tr_feats,
    tr.koht as koht,
    tr.elus as elus

FROM 

transaction_head as tbl1

join 

transaction_v2 as tr

on 
    tbl1.id = tr.head_id
    
where
tr.deprel = 'obl'
and 
(INSTR(',' || tr.feats || ',', ',' || 'abl' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'adit' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'all' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'ad' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'el' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'ill' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'in' || ',') > 0
)
""")



CPU times: user 16.5 s, sys: 4.11 s, total: 20.6 s
Wall time: 34.7 s


#### alustabelisse juurde veergu 'kaane', mis käändega on tegu

In [188]:
cur.execute("""
ALTER TABLE transactions_verbs_obl_kohakaandes_wcomps
ADD kaane VARCHAR(50)
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_wcomps
SET kaane = 'abl'
WHERE INSTR(',' || tr_feats || ',', ',' || 'abl' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_wcomps
SET kaane = 'adit'
WHERE INSTR(',' || tr_feats || ',', ',' || 'adit' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_wcomps
SET kaane = 'all'
WHERE INSTR(',' || tr_feats || ',', ',' || 'all' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_wcomps
SET kaane = 'ad'
WHERE INSTR(',' || tr_feats || ',', ',' || 'ad' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_wcomps
SET kaane = 'el'
WHERE INSTR(',' || tr_feats || ',', ',' || 'el' || ',') > 0
""")
con.commit()


cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_wcomps
SET kaane = 'ill'
WHERE INSTR(',' || tr_feats || ',', ',' || 'ill' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_wcomps
SET kaane = 'in'
WHERE INSTR(',' || tr_feats || ',', ',' || 'in' || ',') > 0
""")
con.commit()


#### vaade mis on tabelis

In [189]:
tables = cur.execute("""
SELECT * FROM transactions_verbs_obl_kohakaandes_wcomps
limit 10
""")

for i, elem in enumerate(tables):
    print(elem)

(2, 'toimuma', '', 1, 'lõpp', 'obl', 'S', 'com,in,sg', 'UNK', 'UNK', 'in')
(3, 'saama', 'pihta', 7, 'keel', 'obl', 'S', 'all,com,pl', 'UNK', 'UNK', 'all')
(10, 'tulema', '', 19, 'sina', 'obl', 'P', 'ad,sg', 'UNK', 'YES', 'ad')
(11, 'viilima', '', 22, 'tund', 'obl', 'S', 'com,el,pl', 'UNK', 'UNK', 'el')
(11, 'viilima', '', 23, 'juht', 'obl', 'S', 'ad,com,sg', 'UNK', 'YES', 'ad')
(25, 'muutuma', '', 40, 'mis', 'obl', 'P', 'el,sg', 'UNK', 'UNK', 'el')
(33, 'minema', 'peale', 59, 'rahvas', 'obl', 'S', 'all,com,sg', 'UNK', 'UNK', 'all')
(37, 'tekkima', '', 69, 'see', 'obl', 'P', 'el,sg', 'UNK', 'UNK', 'el')
(51, 'kutsuma', '', 85, 'elu', 'obl', 'S', 'adit,com,sg', 'UNK', 'UNK', 'adit')
(53, 'tulema', '', 88, 'toim', 'obl', 'S', 'adit,com,sg', 'UNK', 'UNK', 'adit')


### base tabel, kus on distinct verbid eelmisest tabelist ja iga kohakäände jaoks count veerg 

### distinct root count

In [31]:
%%time

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl1
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl1 as
SELECT distinct verb,verb_compound, count(distinct root_word) as abl_cnt
FROM 
transactions_verbs_obl_kohakaandes_wcomps
where INSTR(',' || tr_feats || ',', ',' || 'abl' || ',') > 0
group by verb, verb_compound
""")


cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl2
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl2 as
SELECT distinct verb, verb_compound, count(distinct root_word) as adit_cnt
FROM 
transactions_verbs_obl_kohakaandes_wcomps
where INSTR(',' || tr_feats || ',', ',' || 'adit' || ',') > 0
group by verb, verb_compound
""")



cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl3
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl3 as
SELECT distinct verb, verb_compound, count(distinct root_word) as all_cnt
FROM 
transactions_verbs_obl_kohakaandes_wcomps
where INSTR(',' || tr_feats || ',', ',' || 'all' || ',') > 0
group by verb, verb_compound
""")



cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl4
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl4 as
SELECT distinct verb,verb_compound,  count(distinct root_word) as ad_cnt
FROM 
transactions_verbs_obl_kohakaandes_wcomps
where INSTR(',' || tr_feats || ',', ',' || 'ad' || ',') > 0
group by verb, verb_compound
""")


cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl5
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl5 as
SELECT distinct verb,verb_compound,  count(distinct root_word) as el_cnt
FROM 
transactions_verbs_obl_kohakaandes_wcomps
where INSTR(',' || tr_feats || ',', ',' || 'el' || ',') > 0
group by verb, verb_compound
""")


cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl6
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl6 as
SELECT distinct verb, verb_compound, count(distinct root_word) as ill_cnt
FROM 
transactions_verbs_obl_kohakaandes_wcomps
where INSTR(',' || tr_feats || ',', ',' || 'ill' || ',') > 0
group by verb, verb_compound
""")


cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl7
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl7 as
SELECT distinct verb,verb_compound, count(distinct root_word) as in_cnt
FROM 
transactions_verbs_obl_kohakaandes_wcomps
where INSTR(',' || tr_feats || ',', ',' || 'in' || ',') > 0
group by verb, verb_compound
""")

CPU times: user 13.5 s, sys: 954 ms, total: 14.5 s
Wall time: 14.5 s


In [32]:
%%time

cur.execute("""DROP table if exists transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j1""")
cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j1 AS
select tbl1.verb, tbl1.verb_compound, abl_cnt, adit_cnt from 
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl1 as tbl1
left join
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl2 as tbl2
on tbl1.verb=tbl2.verb 
and tbl1.verb_compound=tbl2.verb_compound
""")

cur.execute("""DROP table if exists transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j2""")
cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j2 AS
select tbl1.verb, tbl1.verb_compound, abl_cnt, adit_cnt, all_cnt from 
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j1 as tbl1
left join
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl3 as tbl2
on tbl1.verb=tbl2.verb 
and tbl1.verb_compound=tbl2.verb_compound
""")


cur.execute("""DROP table if exists transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j3""")
cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j3 AS
select tbl1.verb, tbl1.verb_compound, abl_cnt, adit_cnt, all_cnt, ad_cnt from 
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j2 as tbl1
left join
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl4 as tbl2
on tbl1.verb=tbl2.verb 
and tbl1.verb_compound=tbl2.verb_compound
""")


cur.execute("""DROP table if exists transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j4""")
cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j4 AS
select tbl1.verb, tbl1.verb_compound, 
abl_cnt, adit_cnt, all_cnt, ad_cnt, el_cnt from 
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j3 as tbl1
left join
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl5 as tbl2
on tbl1.verb=tbl2.verb 
and tbl1.verb_compound=tbl2.verb_compound
""")


cur.execute("""DROP table if exists transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j5""")
cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j5 AS
select tbl1.verb, tbl1.verb_compound, 
abl_cnt, adit_cnt, all_cnt, ad_cnt, el_cnt, ill_cnt from 
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j4 as tbl1
left join
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl6 as tbl2
on tbl1.verb=tbl2.verb 
and tbl1.verb_compound=tbl2.verb_compound
""")


cur.execute("""DROP table if exists transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j6""")
cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j6 AS
select tbl1.verb, tbl1.verb_compound,
abl_cnt, adit_cnt, all_cnt, ad_cnt, el_cnt, ill_cnt, in_cnt from 
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j5 as tbl1
left join
transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_tbl7 as tbl2
on tbl1.verb=tbl2.verb 
and tbl1.verb_compound=tbl2.verb_compound
""")




CPU times: user 63.9 ms, sys: 5.98 ms, total: 69.9 ms
Wall time: 103 ms


In [41]:
%%time

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_root_counts_distinct_wcomps_v1 AS
select * from transactions_verbs_obl_kohakaandes_root_counts_base_wcomps_j6
""")

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_root_counts_distinct_wcomps_v1
SET abl_cnt = 0
where abl_cnt is null
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_root_counts_distinct_wcomps_v1
SET adit_cnt = 0
where adit_cnt is null
""")
con.commit()


cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_root_counts_distinct_wcomps_v1
SET all_cnt = 0
where all_cnt is null
""")
con.commit()


cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_root_counts_distinct_wcomps_v1
SET ad_cnt = 0
where ad_cnt is null
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_root_counts_distinct_wcomps_v1
SET el_cnt = 0
where el_cnt is null
""")
con.commit()


cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_root_counts_distinct_wcomps_v1
SET ill_cnt = 0
where ill_cnt is null
""")
con.commit()


cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_root_counts_distinct_wcomps_v1
SET in_cnt = 0
where in_cnt is null
""")
con.commit()

CPU times: user 7.06 ms, sys: 6.03 ms, total: 13.1 ms
Wall time: 34.5 ms


In [42]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_root_counts_distinct_wcomps_v1
"""

source = pd.read_sql_query(query, con)
source

,verb,verb_compound,abl_cnt,adit_cnt,all_cnt,ad_cnt,el_cnt,ill_cnt,in_cnt
0,12olema,,1,0,0,0,0,0,0
1,A. tihkama,,1,0,0,0,0,0,0
2,J. teppima,,1,0,0,0,0,0,0
3,Liitootama,,1,0,0,1,0,0,0
4,Southolema,,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
5651,ütlema,ära,5,2,138,67,488,3,54
5652,üürima,,69,18,107,67,49,9,153
5653,üürima,kokku,1,0,0,0,0,0,0
5654,šantažeerima,välja,1,0,0,0,0,0,0


## 2. tabel

### iga verb+obl+kohakääne jaoks count elus ja count koht, count kokku

1) count distinct root

2) count matches

## base tabel kus on verb, kääne, elus_cnt, koht_cnt, distinct root count  

In [3]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl1

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl1 AS
select 
    verb,
    verb_compound,
    kaane,
    count(distinct root_word) as root_cnt
from (
SELECT 
base.verb, 
base.verb_compound, 
base.kaane, 
verbtbl.root_word, 
verbtbl.koht, 
verbtbl.elus
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps as base
join 
transactions_verbs_obl_kohakaandes_wcomps as verbtbl
on base.verb = verbtbl.verb
and base.verb_compound = verbtbl.verb_compound
and base.kaane = verbtbl.kaane) as tbl1
group by verb, verb_compound, kaane
order by root_cnt desc
""")

CPU times: user 16.3 s, sys: 13.8 s, total: 30.1 s
Wall time: 30.1 s


In [4]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl2

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl2 AS
select 
    verb,
    verb_compound,
    kaane,
    count(distinct root_word) as elus_cnt
from (
SELECT 
base.verb, 
base.verb_compound, 
base.kaane, 
verbtbl.root_word, 
verbtbl.koht, 
verbtbl.elus
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps as base
join 
transactions_verbs_obl_kohakaandes_wcomps as verbtbl
on base.verb = verbtbl.verb
and base.verb_compound = verbtbl.verb_compound
and base.kaane = verbtbl.kaane) as tbl1
where elus='YES'
group by verb, verb_compound, kaane
""")

CPU times: user 1.61 s, sys: 116 ms, total: 1.73 s
Wall time: 1.74 s


In [5]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl3

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl3 AS
select 
    verb,
    verb_compound,
    kaane,
    count(distinct root_word) as koht_cnt
from (
SELECT 
base.verb, 
base.verb_compound, 
base.kaane, 
verbtbl.root_word, 
verbtbl.koht, 
verbtbl.elus
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps as base
join 
transactions_verbs_obl_kohakaandes_wcomps as verbtbl
on base.verb = verbtbl.verb
and base.verb_compound = verbtbl.verb_compound
and base.kaane = verbtbl.kaane) as tbl1
where koht='YES'
group by verb, verb_compound, kaane
""")

CPU times: user 1.16 s, sys: 104 ms, total: 1.27 s
Wall time: 1.27 s


In [4]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_temp1

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_temp1 AS

select tbl1.verb, tbl1.verb_compound, tbl1.kaane, elus_cnt, root_cnt from 
transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl1 as tbl1

left join

transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl2 as tbl2

on tbl1.verb=tbl2.verb 
and tbl1.verb_compound=tbl2.verb_compound
and tbl1.kaane = tbl2.kaane
""")


%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1 AS

select tbl1.verb, tbl1.verb_compound, tbl1.kaane, elus_cnt, koht_cnt, root_cnt from 
transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_temp1 as tbl1

left join

transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl3 as tbl2

on tbl1.verb=tbl2.verb 
and tbl1.verb_compound=tbl2.verb_compound
and tbl1.kaane = tbl2.kaane
""")

CPU times: user 67.9 ms, sys: 4.38 ms, total: 72.2 ms
Wall time: 79.5 ms


In [14]:
cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1
SET elus_cnt = 0
where elus_cnt is null
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1
SET koht_cnt = 0
where koht_cnt is null
""")
con.commit()


In [43]:
%%time

query = """
select * from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1
"""

source = pd.read_sql_query(query, con)
source

CPU times: user 96.2 ms, sys: 2.89 ms, total: 99 ms
Wall time: 99.5 ms


,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt
0,saama,,el,1274,402,19632
1,andma,,all,1601,250,11612
2,rääkima,,el,802,202,10891
3,saama,,in,134,306,8532
4,tulema,,ad,1134,166,8468
...,...,...,...,...,...,...
74715,šveitsima,,el,0,0,1
74716,švipsima,,ad,0,0,1
74717,žestikuleerima,,ad,0,1,1
74718,žisraelima,,ad,0,0,1


In [45]:
source.to_csv('transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_wcomps.csv', index=False, sep=",", encoding="utf-8")

In [46]:
con.close()

# SIIT EDASI ON TABELID, KUS EI OLE VERBIDEGA COMPOUNDE

## 1. tabel 
## verb -> palju esineb obl+kääne (6 kohakäänet) : mitu matchi ja mitu distinct root 

Võtta välja kõik mis on kohakäändes ja sõna deprel on obl

#### alustabel, kus on ainult obl ja kohakäänetes verbid  !!!!!!!!!! pole verb comp

In [73]:
%%time

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes as
SELECT distinct
    tr.head_id as head_id,
    tbl1.verb as verb,
    tr.id as transaction_id,
    tr.lemma as root_word,
    tr.deprel as word_deprel,
    tr.pos as pos,
    tr.feats as tr_feats,
    tr.koht as koht,
    tr.elus as elus

FROM 

transaction_head as tbl1

join 

transaction_v2 as tr

on 
    tbl1.id = tr.head_id
    
where
tr.deprel = 'obl'
and 
(INSTR(',' || tr.feats || ',', ',' || 'abl' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'adit' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'all' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'ad' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'el' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'ill' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'in' || ',') > 0
)
""")



CPU times: user 18 s, sys: 4.13 s, total: 22.1 s
Wall time: 37.7 s


#### alustabelisse juurde veergu 'kaane', mis käändega on tegu

In [74]:
cur.execute("""
ALTER TABLE transactions_verbs_obl_kohakaandes
ADD kaane VARCHAR(50)
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'abl'
WHERE INSTR(',' || tr_feats || ',', ',' || 'abl' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'adit'
WHERE INSTR(',' || tr_feats || ',', ',' || 'adit' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'all'
WHERE INSTR(',' || tr_feats || ',', ',' || 'all' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'ad'
WHERE INSTR(',' || tr_feats || ',', ',' || 'ad' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'el'
WHERE INSTR(',' || tr_feats || ',', ',' || 'el' || ',') > 0
""")
con.commit()


cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'ill'
WHERE INSTR(',' || tr_feats || ',', ',' || 'ill' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'in'
WHERE INSTR(',' || tr_feats || ',', ',' || 'in' || ',') > 0
""")
con.commit()


#### vaade mis on tabelis

In [31]:
tables = cur.execute("""
SELECT * FROM transactions_verbs_obl_kohakaandes
limit 10
""")

for i, elem in enumerate(tables):
    print(elem)

(2, 'toimuma', 1, 'lõpp', 'obl', 'S', 'com,in,sg', 'UNK', 'UNK', 'in')
(3, 'saama', 7, 'keel', 'obl', 'S', 'all,com,pl', 'UNK', 'UNK', 'all')
(10, 'tulema', 19, 'sina', 'obl', 'P', 'ad,sg', 'UNK', 'YES', 'ad')
(11, 'viilima', 22, 'tund', 'obl', 'S', 'com,el,pl', 'UNK', 'UNK', 'el')
(11, 'viilima', 23, 'juht', 'obl', 'S', 'ad,com,sg', 'UNK', 'YES', 'ad')
(25, 'muutuma', 40, 'mis', 'obl', 'P', 'el,sg', 'UNK', 'UNK', 'el')
(33, 'minema', 59, 'rahvas', 'obl', 'S', 'all,com,sg', 'UNK', 'UNK', 'all')
(37, 'tekkima', 69, 'see', 'obl', 'P', 'el,sg', 'UNK', 'UNK', 'el')
(51, 'kutsuma', 85, 'elu', 'obl', 'S', 'adit,com,sg', 'UNK', 'UNK', 'adit')
(53, 'tulema', 88, 'toim', 'obl', 'S', 'adit,com,sg', 'UNK', 'UNK', 'adit')


#### base tabel, kus on distinct verbid eelmisest tabelist ja iga kohakäände jaoks count veerg algväärtusega 0

In [3]:
%%time

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_base
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes_root_counts_base as
SELECT distinct verb, 
0 as abl_cnt, 
0 as adit_cnt, 
0 as all_cnt, 
0 as ad_cnt, 
0 as el_cnt, 
0 as ill_cnt, 
0 as in_cnt

FROM 

transactions_verbs_obl_kohakaandes

""")


CPU times: user 2.05 s, sys: 111 ms, total: 2.16 s
Wall time: 2.21 s


### Täida tabel distinct root countidega

In [4]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_root_counts_base
"""

source = pd.read_sql_query(query, con)

In [5]:
for i in tqdm(range(len(source))):
    verb = source.iloc[i]["verb"]
    #print(verb)
    
    for case in ["abl", "adit", "all", "ad", "el", "ill", "in"]:
        count = 0
        res = cur.execute("""
                select count(distinct root_word) from transactions_verbs_obl_kohakaandes
                where verb='{v}'
                and INSTR(',' || tr_feats || ',', ',' || '{c}' || ',') > 0
                """.format(v=verb, c=case))
        for e in res:
            count = e[0]
        #print(count)
        source.at[i, case+'_cnt'] = count
    #break

100%|█████████████████████████████████████| 9618/9618 [6:42:48<00:00,  2.51s/it]


In [6]:
source

,verb,abl_cnt,adit_cnt,all_cnt,ad_cnt,el_cnt,ill_cnt,in_cnt
0,toimuma,217,314,1026,2953,1076,164,6747
1,saama,5841,1510,6175,5967,21132,1263,9107
2,tulema,3199,2242,7121,9166,9139,2353,6342
3,viilima,2,0,3,16,29,1,14
4,muutuma,166,152,1180,1450,1288,121,2097
...,...,...,...,...,...,...,...,...
9613,lastnuma,0,0,0,0,0,0,1
9614,naajuma,0,0,0,0,0,0,1
9615,sekskima,0,0,0,1,0,0,0
9616,ampima,0,0,0,0,1,0,0


In [13]:
# salvesta

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_distinct
""")

source.to_sql(name='transactions_verbs_obl_kohakaandes_root_counts_distinct', con=con)

source.to_csv("transactions_verbs_obl_kohakaandes_root_counts_distinct.csv", index=False, encoding="utf-8", sep=",")

### Täida tabel root matchide countidega

In [32]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_root_counts_base
"""

source = pd.read_sql_query(query, con)

In [21]:
for i in tqdm(range(len(source))):
    verb = source.iloc[i]["verb"]
    #print(verb)
    
    for case in ["abl", "adit", "all", "ad", "el", "ill", "in"]:
        count = 0
        res = cur.execute("""
                select count(root_word) from transactions_verbs_obl_kohakaandes
                where verb='{v}'
                and INSTR(',' || tr_feats || ',', ',' || '{c}' || ',') > 0
                """.format(v=verb, c=case))
        for e in res:
            count = e[0]
        #print(count)
        source.at[i, case+'_cnt'] = count
    #break

100%|█████████████████████████████████████| 9618/9618 [6:18:48<00:00,  2.36s/it]


In [22]:
source

,verb,abl_cnt,adit_cnt,all_cnt,ad_cnt,el_cnt,ill_cnt,in_cnt
0,toimuma,337,676,2127,51404,3077,274,50217
1,saama,25657,24535,33372,83941,111688,4723,64109
2,tulema,11916,34603,55260,104553,44188,15199,36534
3,viilima,3,0,3,19,37,1,15
4,muutuma,304,201,2991,8985,3580,168,7416
...,...,...,...,...,...,...,...,...
9613,lastnuma,0,0,0,0,0,0,1
9614,naajuma,0,0,0,0,0,0,1
9615,sekskima,0,0,0,1,0,0,0
9616,ampima,0,0,0,0,1,0,0


In [23]:
cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_matches
""")

source.to_sql(name='transactions_verbs_obl_kohakaandes_root_counts_matches', con=con)

source.to_csv("transactions_verbs_obl_kohakaandes_root_counts_matches.csv", index=False, encoding="utf-8", sep=",")

## 2. tabel

### iga verb+obl+kohakääne jaoks count elus ja count koht, count kokku

1) count distinct root

2) count matches

## base tabel kus on verb, kääne, elus_cnt, koht_cnt, distinct root count  

In [3]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base AS
select distinct verb,kaane, verb||'_'||kaane as verb_kaane,
0 as elus_cnt,
0 as koht_cnt,
count(distinct root_word) as root_cnt
from transactions_verbs_obl_kohakaandes
group by verb,kaane
order by root_cnt desc
--order by verb
""")

CPU times: user 6.37 s, sys: 318 ms, total: 6.69 s
Wall time: 6.7 s


In [7]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base
"""

source = pd.read_sql_query(query, con)
source

,verb,kaane,verb_kaane,elus_cnt,koht_cnt,root_cnt
0,saama,el,saama_el,0,0,21132
1,andma,all,andma_all,0,0,12526
2,rääkima,el,rääkima_el,0,0,10930
3,jääma,el,jääma_el,0,0,9936
4,tulema,ad,tulema_ad,0,0,9166
...,...,...,...,...,...,...
30081,šveitsima,el,šveitsima_el,0,0,1
30082,švipsima,ad,švipsima_ad,0,0,1
30083,žestikuleerima,ad,žestikuleerima_ad,0,0,1
30084,žisraelima,ad,žisraelima_ad,0,0,1


In [8]:
for i in tqdm(range(len(source))):
    verb = source.iloc[i]["verb"]
    kaane = source.iloc[i]["kaane"]
    #print(verb)
    
    for case in ["elus", "koht"]:
        count = 0
        res = cur.execute("""
                select count(distinct root_word) from transactions_verbs_obl_kohakaandes
                where verb='{v}'
                and kaane = '{k}'
                and {c}='YES'
                """.format(v=verb, k=kaane, c=case))
        for e in res:
            count = e[0]
        #print(count)
        source.at[i, case+'_cnt'] = count
    #break

100%|███████████████████████████████████| 30086/30086 [6:08:20<00:00,  1.36it/s]


In [9]:
source

,verb,kaane,verb_kaane,elus_cnt,koht_cnt,root_cnt
0,saama,el,saama_el,1317,408,21132
1,andma,all,andma_all,1649,267,12526
2,rääkima,el,rääkima_el,805,202,10930
3,jääma,el,jääma_el,709,265,9936
4,tulema,ad,tulema_ad,1178,174,9166
...,...,...,...,...,...,...
30081,šveitsima,el,šveitsima_el,0,0,1
30082,švipsima,ad,švipsima_ad,0,0,1
30083,žestikuleerima,ad,žestikuleerima_ad,0,1,1
30084,žisraelima,ad,žisraelima_ad,0,0,1


In [10]:
cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct
""")

source.to_sql(name='transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct', con=con)

## verb comp versioon

In [3]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_wcomps
limit 10
"""

source2 = pd.read_sql_query(query, con)
source2

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,kaane
0,2,toimuma,,1,lõpp,obl,S,"com,in,sg",UNK,UNK,in
1,3,saama,pihta,7,keel,obl,S,"all,com,pl",UNK,UNK,all
2,10,tulema,,19,sina,obl,P,"ad,sg",UNK,YES,ad
3,11,viilima,,22,tund,obl,S,"com,el,pl",UNK,UNK,el
4,11,viilima,,23,juht,obl,S,"ad,com,sg",UNK,YES,ad
5,25,muutuma,,40,mis,obl,P,"el,sg",UNK,UNK,el
6,33,minema,peale,59,rahvas,obl,S,"all,com,sg",UNK,UNK,all
7,37,tekkima,,69,see,obl,P,"el,sg",UNK,UNK,el
8,51,kutsuma,,85,elu,obl,S,"adit,com,sg",UNK,UNK,adit
9,53,tulema,,88,toim,obl,S,"adit,com,sg",UNK,UNK,adit


In [3]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl1

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl1 AS
select 
    verb,
    verb_compound,
    kaane,
    count(distinct root_word) as root_cnt
from (
SELECT 
base.verb, 
base.verb_compound, 
base.kaane, 
verbtbl.root_word, 
verbtbl.koht, 
verbtbl.elus
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps as base
join 
transactions_verbs_obl_kohakaandes_wcomps as verbtbl
on base.verb = verbtbl.verb
and base.verb_compound = verbtbl.verb_compound
and base.kaane = verbtbl.kaane) as tbl1
group by verb, verb_compound, kaane
order by root_cnt desc
""")

CPU times: user 16.3 s, sys: 13.8 s, total: 30.1 s
Wall time: 30.1 s


In [4]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl2

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl2 AS
select 
    verb,
    verb_compound,
    kaane,
    count(distinct root_word) as elus_cnt
from (
SELECT 
base.verb, 
base.verb_compound, 
base.kaane, 
verbtbl.root_word, 
verbtbl.koht, 
verbtbl.elus
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps as base
join 
transactions_verbs_obl_kohakaandes_wcomps as verbtbl
on base.verb = verbtbl.verb
and base.verb_compound = verbtbl.verb_compound
and base.kaane = verbtbl.kaane) as tbl1
where elus='YES'
group by verb, verb_compound, kaane
""")

CPU times: user 1.61 s, sys: 116 ms, total: 1.73 s
Wall time: 1.74 s


In [5]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl3

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl3 AS
select 
    verb,
    verb_compound,
    kaane,
    count(distinct root_word) as koht_cnt
from (
SELECT 
base.verb, 
base.verb_compound, 
base.kaane, 
verbtbl.root_word, 
verbtbl.koht, 
verbtbl.elus
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps as base
join 
transactions_verbs_obl_kohakaandes_wcomps as verbtbl
on base.verb = verbtbl.verb
and base.verb_compound = verbtbl.verb_compound
and base.kaane = verbtbl.kaane) as tbl1
where koht='YES'
group by verb, verb_compound, kaane
""")

CPU times: user 1.16 s, sys: 104 ms, total: 1.27 s
Wall time: 1.27 s


In [4]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_temp1

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_temp1 AS

select tbl1.verb, tbl1.verb_compound, tbl1.kaane, elus_cnt, root_cnt from 
transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl1 as tbl1

left join

transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl2 as tbl2

on tbl1.verb=tbl2.verb 
and tbl1.verb_compound=tbl2.verb_compound
and tbl1.kaane = tbl2.kaane
""")

CPU times: user 67.9 ms, sys: 4.38 ms, total: 72.2 ms
Wall time: 79.5 ms


In [8]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1 AS

select tbl1.verb, tbl1.verb_compound, tbl1.kaane, elus_cnt, koht_cnt, root_cnt from 
transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_temp1 as tbl1

left join

transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_tbl3 as tbl2

on tbl1.verb=tbl2.verb 
and tbl1.verb_compound=tbl2.verb_compound
and tbl1.kaane = tbl2.kaane
""")

CPU times: user 68 ms, sys: 7.54 ms, total: 75.6 ms
Wall time: 93.2 ms


In [14]:
cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1
SET elus_cnt = 0
where elus_cnt is null
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1
SET koht_cnt = 0
where koht_cnt is null
""")
con.commit()


In [18]:
%%time

query = """
select * from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base_wcomps_v1
"""

source = pd.read_sql_query(query, con)
source

CPU times: user 95.6 ms, sys: 5.5 ms, total: 101 ms
Wall time: 101 ms


,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt
0,saama,,el,1274,402,19632
1,andma,,all,1601,250,11612
2,rääkima,,el,802,202,10891
3,saama,,in,134,306,8532
4,tulema,,ad,1134,166,8468
...,...,...,...,...,...,...
74715,šveitsima,,el,0,0,1
74716,švipsima,,ad,0,0,1
74717,žestikuleerima,,ad,0,1,1
74718,žisraelima,,ad,0,0,1


## base tabel kus on verb, kääne, elus_cnt, koht_cnt, root count  

In [3]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches_base

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches_base AS
select distinct verb,kaane, verb||'_'||kaane as verb_kaane,
0 as elus_cnt,
0 as koht_cnt,
count(root_word) as root_cnt
from transactions_verbs_obl_kohakaandes
group by verb,kaane
order by root_cnt desc
--order by verb
""")

CPU times: user 4.69 s, sys: 456 ms, total: 5.14 s
Wall time: 8.82 s


In [33]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches_base
"""

source = pd.read_sql_query(query, con)


In [8]:
for i in tqdm(range(len(source))):
    verb = source.iloc[i]["verb"]
    kaane = source.iloc[i]["kaane"]
    #print(verb)
    
    for case in ["elus", "koht"]:
        count = 0
        res = cur.execute("""
                select count(root_word) from transactions_verbs_obl_kohakaandes
                where verb='{v}'
                and kaane = '{k}'
                and {c}='YES'
                """.format(v=verb, k=kaane, c=case))
        for e in res:
            count = e[0]
        #print(count)
        source.at[i, case+'_cnt'] = count
    #break

100%|███████████████████████████████████| 30086/30086 [6:09:06<00:00,  1.36it/s]


In [9]:
source

,verb,kaane,verb_kaane,elus_cnt,koht_cnt,root_cnt
0,saama,el,saama_el,13448,5330,111688
1,tulema,ad,tulema_ad,24980,3147,104553
2,andma,all,andma_all,34988,3385,89057
3,saama,ad,saama_ad,4303,2893,83941
4,olema,ad,olema_ad,22693,4253,75894
...,...,...,...,...,...,...
30081,šveitsima,el,šveitsima_el,0,0,1
30082,švipsima,ad,švipsima_ad,0,0,1
30083,žestikuleerima,ad,žestikuleerima_ad,0,1,1
30084,žisraelima,ad,žisraelima_ad,0,0,1


In [10]:
cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches
""")

source.to_sql(name='transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches', con=con)

In [47]:
con.close()